# Spatial neighbourhood analysis

This notebook computes and plots Delaunay-graph spatial-neighbourhood enrichment
(`squidpy.gr.nhood_enrichment`) between annotated cell populations in the Xenium dataset.

The core four-step pattern — subset one sample, build its spatial-neighbour graph, compute
enrichment, plot and save the heatmap — is explained once (for `BE_rep1`) and then repeated
for every other replicate without re-explaining each step.


## Setup

Imports, plotting defaults, and output paths used throughout the notebook.


In [ ]:
from spatialdata_io import xenium
from pathlib import Path
import logging


In [ ]:
import spatialdata as sd

import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import squidpy as sq
import numpy as np
import pandas as pd


In [ ]:
import matplotlib as mpl

# Keep text in exported PDFs as real, editable text (not outlined paths) when opened in
# Illustrator, by embedding TrueType fonts instead of converting text to vector shapes.
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"]  = 42


In [ ]:
# Silence noisy font-subsetting debug logs that matplotlib/fontTools emit when saving PDFs.
logging.getLogger("fontTools").setLevel(logging.WARNING)
logging.getLogger("fontTools.subset").setLevel(logging.WARNING)


In [ ]:
# Default directory scanpy writes to when a plotting function is called with save=... .
sc.settings.figdir = "../Figures_forpaper/neighbourhood_noWT3"

# Default resolution/background used for figures saved that way.
sc.settings.set_figure_params(dpi_save=300, transparent=True)


In [ ]:
# Directory used below for figures saved manually via plt.savefig(); create it if missing.
out_dir = Path("../Figures_forpaper/neighbourhood_noWT3")
out_dir.mkdir(parents=True, exist_ok=True)


## Load processed spatial data

Load the annotated, clustered AnnData object produced by the upstream processing/clustering
notebookand confirm it looks as expected.


In [ ]:
# Load the annotated, clustered AnnData object.
adata = sc.read_h5ad(
    "combined_Xenium_largecells_noWT3_afterclustering_withleiden_annotated_ordered_withTcells_ReannotatedIfgga4.h5ad"
)


In [ ]:
# Inspect the loaded object: dimensions, obs/var columns, and stored layers/embeddings.
adata


## Fine-grained cell-type order (`celltype4`)

Fix a consistent display order for the fine-grained cell-type annotation (`celltype4`) so that
every neighbourhood-enrichment heatmap below shows populations in the same order and is
therefore directly comparable across samples.


In [ ]:
# Number of cells per fine-grained cell type across the full (all-sample) dataset.
adata.obs['celltype4'].value_counts()


In [ ]:
# Fixed display order for the fine cell-type categories, used consistently across all
# neighbourhood-enrichment heatmaps below.
order = ['Ifgga4+ VCMs', 'Lymphoid', 'Myeloid',  'Pericytes', 'VCMs',  'Vasculature ECs', 'Stressed VCMs', 
         'FBs', 'Endocardial ECs', 'Myh7+ VCMs', 'Epicardium', 'SMCs', 'NCs',  'ACMs']

adata.obs['celltype4'] = adata.obs['celltype4'].astype("category")
adata.obs['celltype4'] = adata.obs['celltype4'].cat.reorder_categories(order, ordered=True)


### Worked example: Base-Edited replicate 1

The next several cells work through the full per-replicate pipeline once, for `BE_rep1`:
subset the sample, build its spatial-neighbour graph, compute neighbourhood enrichment, and
export the heatmap — plus two extras specific to this replicate: a sensitivity check with the
rare ACM population excluded, and a spatial QC plot. Later replicates reuse the same core
subset → graph → enrichment → plot pattern without repeating the explanation.


In [ ]:
# Work on one replicate at a time: subset the combined object to this sample.
BE_rep1 = adata[adata.obs['name']=='BE_rep1'].copy()


In [ ]:
BE_rep1.obs['celltype4'].value_counts()


In [ ]:
# Subsetting an AnnData resets categorical ordering to whatever categories are present, so
# re-apply the same fixed order used above to this sample's copy.
order = ['Ifgga4+ VCMs', 'Lymphoid', 'Myeloid',  'Pericytes', 'VCMs',  'Vasculature ECs', 'Stressed VCMs', 
         'FBs', 'Endocardial ECs', 'Myh7+ VCMs', 'Epicardium', 'SMCs', 'NCs',  'ACMs']

BE_rep1.obs['celltype4'] = BE_rep1.obs['celltype4'].astype("category")
BE_rep1.obs['celltype4'] = BE_rep1.obs['celltype4'].cat.reorder_categories(order, ordered=True)


## Construct the spatial neighbourhood graph

Define which cells are spatial neighbours using a Delaunay triangulation on cell centroids
(`squidpy.gr.spatial_neighbors`). The resulting graph is the basis for the neighbourhood
enrichment test below.


In [ ]:
# Delaunay triangulation on cell centroids defines which cells count as spatial neighbours.
sq.gr.spatial_neighbors(BE_rep1, coord_type="generic", delaunay=True)


## Neighbourhood enrichment

Test whether pairs of annotated cell populations occur next to one another more or less often
than expected by chance, using squidpy's permutation-based neighbourhood-enrichment test.


In [ ]:
# Permutation-based test for over/under-representation of each pair of cell types among
# spatial neighbours (result stored in adata.uns['celltype4_nhood_enrichment']).
sq.gr.nhood_enrichment(BE_rep1, cluster_key="celltype4")


## Export figures

Plot the enrichment z-score matrix as a heatmap and save it as a PDF for the manuscript. This
plot → save → show pattern is reused, unannotated, for every replicate below.


In [ ]:
# Plot the z-scored neighbourhood-enrichment heatmap and save it as a PDF.
sq.pl.nhood_enrichment(
    BE_rep1,
    cluster_key="celltype4",
    figsize=(8, 8),
    title="Neighborhood enrichment (Base-Edited 1)",
    mode='zscore',
    #method="average",
    cmap="seismic",vmin=-100, vmax=100, vcenter=0
)

# Save the heatmap to disk.
plt.savefig(
    out_dir/"BE_rep1_neighbourhood_enrichment.pdf",
    transparent=True,
    bbox_inches="tight",
)

# Also display it inline.
plt.show()


In [ ]:
# Sensitivity check: this replicate contains very few ACM-labelled cells (only 2 — see the
# output filename below), too few for a meaningful pairwise enrichment estimate. Repeat the
# analysis for BE_rep1 with ACMs excluded.
BE_rep1_noACM = BE_rep1[BE_rep1.obs["celltype4"] != "ACMs"].copy()

# The categorical dtype still remembers "ACMs" as a category unless explicitly dropped; remove
# it so it doesn't show up as an empty row/column in the enrichment matrix and heatmap.
BE_rep1_noACM.obs["celltype4"] = BE_rep1_noACM.obs["celltype4"].cat.remove_unused_categories()


In [ ]:
# Same fixed ordering as above, with "ACMs" dropped (it was removed from this subset).
order = ['Ifgga4+ VCMs', 'Lymphoid', 'Myeloid',  'Pericytes', 'VCMs',  'Vasculature ECs', 'Stressed VCMs', 
         'FBs', 'Endocardial ECs', 'Myh7+ VCMs', 'Epicardium', 'SMCs', 'NCs']

BE_rep1_noACM.obs['celltype4'] = BE_rep1_noACM.obs['celltype4'].astype("category")
BE_rep1_noACM.obs['celltype4'] = BE_rep1_noACM.obs['celltype4'].cat.reorder_categories(order, ordered=True)


In [ ]:
# Rebuild the spatial-neighbour graph on the ACM-excluded subset (graph indices must match
# the cells actually present).
sq.gr.spatial_neighbors(BE_rep1_noACM, coord_type="generic", delaunay=True)


In [ ]:
# Recompute neighbourhood enrichment without ACMs.
sq.gr.nhood_enrichment(BE_rep1_noACM, cluster_key="celltype4")


In [ ]:
# Plot and save the ACM-excluded version of the heatmap (filename notes ACMs were only 2 cells).
sq.pl.nhood_enrichment(
    BE_rep1_noACM,
    cluster_key="celltype4",
    figsize=(8, 8),
    title="Neighborhood enrichment (Base-Edited 1)",
    mode='zscore',
    #method="average",
    cmap="seismic",vmin=-100, vmax=100, vcenter=0
)

# Save the heatmap to disk.
plt.savefig(
    out_dir/"BE_rep1_neighbourhood_enrichment_noACMs-were2cells.pdf",
    transparent=True,
    bbox_inches="tight",
)

# Also display it inline.
plt.show()


In [ ]:
# QC/illustrative plot (not saved to disk): where are the Ifgga4+ VCMs located within this
# section, relative to all other cells?
sc.pl.spatial(BE_rep1, color=["celltype4"], groups=["Ifgga4+ VCMs"], na_color = 'lightgrey', spot_size = 25)


### Base-Edited replicates 2–4

Same subset → build graph → compute enrichment → plot/save pattern as `BE_rep1` above, repeated for each remaining Base-Edited replicate.


In [ ]:
# Subset to this replicate.
BE_rep2 = adata[adata.obs['name']=='BE_rep2'].copy()


In [ ]:
# Build the spatial-neighbour graph for this replicate.
sq.gr.spatial_neighbors(BE_rep2, coord_type="generic", delaunay=True)


In [ ]:
# Compute neighbourhood enrichment for this replicate.
sq.gr.nhood_enrichment(BE_rep2, cluster_key="celltype4")


In [ ]:
# Plot the z-scored neighbourhood-enrichment heatmap and save it as a PDF.
sq.pl.nhood_enrichment(
    BE_rep2,
    cluster_key="celltype4",
    figsize=(8, 8),
    title="Neighborhood enrichment (Base-Edited 2)",
    mode='zscore',
    #method="average",
    cmap="seismic",vmin=-100, vmax=100, vcenter=0
)

# Save the heatmap to disk.
plt.savefig(
    out_dir/"BE_rep2_neighbourhood_enrichment.pdf",
    transparent=True,
    bbox_inches="tight",
)

# Also display it inline.
plt.show()


In [ ]:
# Subset to this replicate.
BE_rep3 = adata[adata.obs['name']=='BE_rep3'].copy()


In [ ]:
# Build the spatial-neighbour graph for this replicate.
sq.gr.spatial_neighbors(BE_rep3, coord_type="generic", delaunay=True)


In [ ]:
# Compute neighbourhood enrichment for this replicate.
sq.gr.nhood_enrichment(BE_rep3, cluster_key="celltype4")


In [ ]:
# Quick sanity check on the subset before digging into its enrichment results below.
BE_rep3


In [ ]:
# The enrichment results are stored in .uns as a dict of two square matrices, "zscore" and
# "count" (both ordered like adata.obs['celltype4'].cat.categories).
ne = BE_rep3.uns["celltype4_nhood_enrichment"]
print(type(ne))
print(ne.keys())


In [ ]:
# Raw z-score matrix (rows/columns follow the celltype4 category order set above).
ne["zscore"]


In [ ]:
# Label the raw matrices with cell-type names for readability, and so they can be exported
# as supplementary tables alongside the figures.
cats = BE_rep3.obs["celltype4"].cat.categories
zscore_df = pd.DataFrame(ne["zscore"], index=cats, columns=cats)
count_df = pd.DataFrame(ne["count"], index=cats, columns=cats)
zscore_df


In [ ]:
# Plot and save the standard heatmap for BE_rep3 (same style as the other replicates).
sq.pl.nhood_enrichment(
    BE_rep3,
    cluster_key="celltype4",
    figsize=(8, 8),
    title="Neighborhood enrichment (Base-Edited 3)",
    mode='zscore',
    #method="average",
    cmap="seismic",vmin=-100, vmax=100, vcenter=0
)

# Save the heatmap to disk.
plt.savefig(
    out_dir/"BE_rep3_neighbourhood_enrichment.pdf",
    transparent=True,
    bbox_inches="tight",
)

# Also display it inline.
plt.show()


In [ ]:
# Alternative, publication-formatted version of the panel above: draw onto an explicit Axes
# so the colorbar ticks can be cleaned up manually.
import matplotlib.ticker as mticker

fig, ax = plt.subplots(figsize=(8, 8))

sq.pl.nhood_enrichment(
    BE_rep3,
    cluster_key="celltype4",
    figsize=(8, 8),
    title="Neighborhood enrichment (Base-Edited 3)",
    mode="zscore",
    cmap="seismic",
    vmin=-100,
    vmax=100,
    vcenter=0,
    ax=ax
)

# squidpy draws several axes for this plot; the continuous colorbar is normally fig.axes[3].
cbar_ax = fig.axes[3]

# Use a fixed set of round tick values instead of squidpy's default colorbar ticks.
cbar_ax.yaxis.set_major_locator(mticker.FixedLocator([-100, -50, 0, 50, 100]))
cbar_ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f"))

plt.savefig(
    out_dir / "BE_rep3_neighbourhood_enrichment_prettier.pdf",
    transparent=True,
    bbox_inches="tight",
)
plt.show()


In [ ]:
# Subset to this replicate.
BE_rep4 = adata[adata.obs['name']=='BE_rep4'].copy()


In [ ]:
# Build the spatial-neighbour graph for this replicate.
sq.gr.spatial_neighbors(BE_rep4, coord_type="generic", delaunay=True)


In [ ]:
# Compute neighbourhood enrichment for this replicate.
sq.gr.nhood_enrichment(BE_rep4, cluster_key="celltype4")


In [ ]:
# Plot the z-scored neighbourhood-enrichment heatmap and save it as a PDF.
sq.pl.nhood_enrichment(
    BE_rep4,
    cluster_key="celltype4",
    figsize=(8, 8),
    title="Neighborhood enrichment (Base-Edited 4)",
    mode='zscore',
    #method="average",
    cmap="seismic",vmin=-100, vmax=100, vcenter=0
)

# Save the heatmap to disk.
plt.savefig(
    out_dir/"BE_rep4_neighbourhood_enrichment.pdf",
    transparent=True,
    bbox_inches="tight",
)

# Also display it inline.
plt.show()


### R636Q (PBS-treated) replicates 1–4

Same pattern again, for the PBS/vehicle-treated R636Q replicates (figure titles read “R636Q N”).


In [ ]:
# Subset to this replicate.
PBS_rep1 = adata[adata.obs['name']=='PBS_rep1'].copy()


In [ ]:
# Build the spatial-neighbour graph for this replicate.
sq.gr.spatial_neighbors(PBS_rep1, coord_type="generic", delaunay=True)


In [ ]:
# Compute neighbourhood enrichment for this replicate.
sq.gr.nhood_enrichment(PBS_rep1, cluster_key="celltype4")


In [ ]:
# Plot the z-scored neighbourhood-enrichment heatmap and save it as a PDF.
sq.pl.nhood_enrichment(
    PBS_rep1,
    cluster_key="celltype4",
    figsize=(8, 8),
    title="Neighborhood enrichment (R636Q 1)",
    mode='zscore',
    #method="average",
    cmap="seismic",vmin=-100, vmax=100, vcenter=0
)

# Save the heatmap to disk.
plt.savefig(
    out_dir/"PBS_rep1_neighbourhood_enrichment.pdf",
    transparent=True,
    bbox_inches="tight",
)

# Also display it inline.
plt.show()


In [ ]:
# Subset to this replicate.
PBS_rep2 = adata[adata.obs['name']=='PBS_rep2'].copy()


In [ ]:
# Build the spatial-neighbour graph for this replicate.
sq.gr.spatial_neighbors(PBS_rep2, coord_type="generic", delaunay=True)


In [ ]:
# Compute neighbourhood enrichment for this replicate.
sq.gr.nhood_enrichment(PBS_rep2, cluster_key="celltype4")


In [ ]:
# Plot the z-scored neighbourhood-enrichment heatmap and save it as a PDF.
sq.pl.nhood_enrichment(
    PBS_rep2,
    cluster_key="celltype4",
    figsize=(8, 8),
    title="Neighborhood enrichment (R636Q 2)",
    mode='zscore',
    #method="average",
    cmap="seismic",vmin=-100, vmax=100, vcenter=0
)

# Save the heatmap to disk.
plt.savefig(
    out_dir/"PBS_rep2_neighbourhood_enrichment.pdf",
    transparent=True,
    bbox_inches="tight",
)

# Also display it inline.
plt.show()


In [ ]:
# Subset to this replicate.
PBS_rep3 = adata[adata.obs['name']=='PBS_rep3'].copy()


In [ ]:
# Build the spatial-neighbour graph for this replicate.
sq.gr.spatial_neighbors(PBS_rep3, coord_type="generic", delaunay=True)


In [ ]:
# Compute neighbourhood enrichment for this replicate.
sq.gr.nhood_enrichment(PBS_rep3, cluster_key="celltype4")


In [ ]:
# Plot the z-scored neighbourhood-enrichment heatmap and save it as a PDF.
sq.pl.nhood_enrichment(
    PBS_rep3,
    cluster_key="celltype4",
    figsize=(8, 8),
    title="Neighborhood enrichment (R636Q 3)",
    mode='zscore',
    #method="average",
    cmap="seismic",vmin=-100, vmax=100, vcenter=0
)

# Save the heatmap to disk.
plt.savefig(
    out_dir/"PBS_rep3_neighbourhood_enrichment.pdf",
    transparent=True,
    bbox_inches="tight",
)

# Also display it inline.
plt.show()


In [ ]:
# Subset to this replicate.
PBS_rep4 = adata[adata.obs['name']=='PBS_rep4'].copy()


In [ ]:
# Build the spatial-neighbour graph for this replicate.
sq.gr.spatial_neighbors(PBS_rep4, coord_type="generic", delaunay=True)


In [ ]:
# Compute neighbourhood enrichment for this replicate.
sq.gr.nhood_enrichment(PBS_rep4, cluster_key="celltype4")


In [ ]:
# Plot the z-scored neighbourhood-enrichment heatmap and save it as a PDF.
sq.pl.nhood_enrichment(
    PBS_rep4,
    cluster_key="celltype4",
    figsize=(8, 8),
    title="Neighborhood enrichment (R636Q 4)",
    mode='zscore',
    #method="average",
    cmap="seismic",vmin=-100, vmax=100, vcenter=0
)

# Save the heatmap to disk.
plt.savefig(
    out_dir/"PBS_rep4_neighbourhood_enrichment.pdf",
    transparent=True,
    bbox_inches="tight",
)

# Also display it inline.
plt.show()


### Wild-type replicates (`WT_rep3` excluded)

Same pattern again, for the wild-type replicates. `WT_rep3` is intentionally absent from this notebook.


In [ ]:
# Subset to this replicate.
WT_rep1 = adata[adata.obs['name']=='WT_rep1'].copy()


In [ ]:
# Build the spatial-neighbour graph for this replicate.
sq.gr.spatial_neighbors(WT_rep1, coord_type="generic", delaunay=True)


In [ ]:
# Compute neighbourhood enrichment for this replicate.
sq.gr.nhood_enrichment(WT_rep1, cluster_key="celltype4")


In [ ]:
# Plot the z-scored neighbourhood-enrichment heatmap and save it as a PDF.
sq.pl.nhood_enrichment(
    WT_rep1,
    cluster_key="celltype4",
    figsize=(8, 8),
    title="Neighborhood enrichment (WT 1)",
    mode='zscore',
    #method="average",
    cmap="seismic",vmin=-100, vmax=100, vcenter=0
)

# Save the heatmap to disk.
plt.savefig(
    out_dir/"WT_rep1_neighbourhood_enrichment.pdf",
    transparent=True,
    bbox_inches="tight",
)

# Also display it inline.
plt.show()


In [ ]:
# Subset to this replicate.
WT_rep2 = adata[adata.obs['name']=='WT_rep2'].copy()


In [ ]:
# Build the spatial-neighbour graph for this replicate.
sq.gr.spatial_neighbors(WT_rep2, coord_type="generic", delaunay=True)


In [ ]:
# Compute neighbourhood enrichment for this replicate.
sq.gr.nhood_enrichment(WT_rep2, cluster_key="celltype4")


In [ ]:
# Plot the z-scored neighbourhood-enrichment heatmap and save it as a PDF.
sq.pl.nhood_enrichment(
    WT_rep2,
    cluster_key="celltype4",
    figsize=(8, 8),
    title="Neighborhood enrichment (WT 2)",
    mode='zscore',
    #method="average",
    cmap="seismic",vmin=-100, vmax=100, vcenter=0
)

# Save the heatmap to disk.
plt.savefig(
    out_dir/"WT_rep2_neighbourhood_enrichment.pdf",
    transparent=True,
    bbox_inches="tight",
)

# Also display it inline.
plt.show()


In [ ]:
# Subset to this replicate.
WT_rep4 = adata[adata.obs['name']=='WT_rep4'].copy()


In [ ]:
# Build the spatial-neighbour graph for this replicate.
sq.gr.spatial_neighbors(WT_rep4, coord_type="generic", delaunay=True)


In [ ]:
# Compute neighbourhood enrichment for this replicate.
sq.gr.nhood_enrichment(WT_rep4, cluster_key="celltype4")


In [ ]:
# Plot the z-scored neighbourhood-enrichment heatmap and save it as a PDF.
sq.pl.nhood_enrichment(
    WT_rep4,
    cluster_key="celltype4",
    figsize=(8, 8),
    title="Neighborhood enrichment (WT 4)",
    mode='zscore',
    #method="average",
    cmap="seismic",vmin=-100, vmax=100, vcenter=0
)

# Save the heatmap to disk.
plt.savefig(
    out_dir/"WT_rep4_neighbourhood_enrichment.pdf",
    transparent=True,
    bbox_inches="tight",
)

# Also display it inline.
plt.show()


In [ ]:
# Subset to this replicate.
WT_rep5 = adata[adata.obs['name']=='WT_rep5'].copy()


In [ ]:
# Build the spatial-neighbour graph for this replicate.
sq.gr.spatial_neighbors(WT_rep5, coord_type="generic", delaunay=True)


In [ ]:
# Compute neighbourhood enrichment for this replicate.
sq.gr.nhood_enrichment(WT_rep5, cluster_key="celltype4")


In [ ]:
# Plot the z-scored neighbourhood-enrichment heatmap and save it as a PDF.
sq.pl.nhood_enrichment(
    WT_rep5,
    cluster_key="celltype4",
    figsize=(8, 8),
    title="Neighborhood enrichment (WT 5)",
    mode='zscore',
    #method="average",
    cmap="seismic",vmin=-100, vmax=100, vcenter=0
)

# Save the heatmap to disk.
plt.savefig(
    out_dir/"WT_rep5_neighbourhood_enrichment.pdf",
    transparent=True,
    bbox_inches="tight",
)

# Also display it inline.
plt.show()
